# The Pareidolia Paradox

Ten complementary representations of each crop, combined by a fold-safe stack.

Every crop is first rotated by **-`sun_azimuth_angle`** so the sun sits in a fixed direction, then
centre-cropped to 224 px and resized to 256 px.

| Branch | Representation | Why |
|---|---|---|
| `hog_norm` / `hog_fine` / `hog_coarse` | **HOG** at 16 / 8 / 32 px cells | gradient structure at three spatial scales |
| `hog_centre` | HOG on the centre 128 px | tests whether the signal is central or peripheral |
| `hog_raw` | HOG on raw crops | RAW vs normalised ablation |
| `lfm` / `lfm_ps8` | NASA-IBM **Lunar Foundation Model** (ViT-B, `nac`), frozen, depths 5/8/11, patch 16 and 8 | lunar-pretrained features at two token resolutions |
| `dino` / `dino_vit` | **DINOv3** ConvNeXt-S and ViT-B, frozen | general-purpose texture/shape features |
| `sunrel` | gradient-orientation histograms per radial ring | with the sun direction fixed, orientation carries the bright/dark ordering |

Every branch is cross-validated with the **same** label x azimuth-stratified folds; their out-of-fold
probabilities are then stacked (fold-safe: the meta-model is itself cross-validated on those folds).
No backbone is fine-tuned, so each one needs a single forward pass and the whole notebook runs in
about 25 minutes on a single T4.

## 1. Install & imports

In [5]:
!pip install -q -U "timm>=1.0.20" "huggingface_hub>=0.24" einops omegaconf hdf5plugin
!git clone -q --depth 1 https://github.com/NASA-IMPACT/NASA-IBM-Lunar-Foundation-Model /kaggle/working/lfm_repo || echo "lfm_repo present"

fatal: destination path '/kaggle/working/lfm_repo' already exists and is not an empty directory.
lfm_repo present


In [6]:
import os, gc, time, zipfile, warnings, importlib.util, sys
from concurrent.futures import ThreadPoolExecutor

import numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt
import torch, torch.nn.functional as F
import timm
from skimage.feature import hog
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

warnings.filterwarnings("ignore"); cv2.setNumThreads(0)
torch.backends.cudnn.benchmark = True
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| timm", timm.__version__, "| device", DEV)

SEED, N_FOLDS = 42, 5
IMG_SIZE, ROT_CROP = 256, 224
AZ_STRAT_BINS = 8
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/input") else "."
LFM_REPO, LFM_WEIGHTS = "/kaggle/working/lfm_repo", "/kaggle/working/lfm_weights"
LFM_HF_REPO = "nasa-ibm-ai4science/NASA-IBM-Lunar-Foundation-Model"
USE_LFM, USE_DINO = True, True          # frozen backbones (need internet); HOG branches always run
np.random.seed(SEED); torch.manual_seed(SEED)

torch 2.10.0+cu128 | timm 1.0.29 | device cuda


## 2. Data, folds, azimuth-normalised crops

In [7]:
SEARCH_ROOTS = [r for r in ["/kaggle/input", OUT_DIR, "."] if os.path.isdir(r)]

def _find(pred):
    for root in SEARCH_ROOTS:
        for dp, dn, fn in os.walk(root):
            r = pred(dp, fn)
            if r:
                return r
    return None

train_df = pd.read_csv(_find(lambda dp, fn: os.path.join(dp, "train_metadata.csv") if "train_metadata.csv" in fn else None))
test_df = pd.read_csv(_find(lambda dp, fn: os.path.join(dp, "test_metadata.csv") if "test_metadata.csv" in fn else None))
train_dir = _find(lambda dp, fn: dp if train_df.image_id.iloc[0] in fn else None)
test_dir = _find(lambda dp, fn: dp if test_df.image_id.iloc[0] in fn else None)
print("train", train_df.shape, train_dir, "| test", test_df.shape, test_dir)

y = train_df.label.values.astype(int)
AZ_TR = train_df.sun_azimuth_angle.values.astype(np.float32)
AZ_TE = test_df.sun_azimuth_angle.values.astype(np.float32)

def read(folder, name):
    im = cv2.imread(os.path.join(folder, name), cv2.IMREAD_GRAYSCALE)
    return im if im.shape == (256, 256) else cv2.resize(im, (256, 256), interpolation=cv2.INTER_AREA)

def load_all(folder, names):
    with ThreadPoolExecutor(8) as ex:
        return np.stack(list(ex.map(lambda n: read(folder, n), names)))

t0 = time.time()
RAW_TR, RAW_TE = load_all(train_dir, train_df.image_id), load_all(test_dir, test_df.image_id)
print("loaded", RAW_TR.shape, RAW_TE.shape, f"{time.time()-t0:.0f}s")

def normalise(imgs, azs, out_size=IMG_SIZE):
    # rotate each crop by -sun_azimuth_angle, centre-crop, resize
    def f(i):
        M = cv2.getRotationMatrix2D((128, 128), float(-azs[i]), 1.0)
        r = cv2.warpAffine(imgs[i], M, (256, 256), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
        s = (256 - ROT_CROP) // 2
        return cv2.resize(r[s:s + ROT_CROP, s:s + ROT_CROP], (out_size, out_size), interpolation=cv2.INTER_LINEAR)
    with ThreadPoolExecutor(8) as ex:
        return np.stack(list(ex.map(f, range(len(imgs)))))

t0 = time.time()
NRM_TR, NRM_TE = normalise(RAW_TR, AZ_TR), normalise(RAW_TE, AZ_TE)
print("azimuth-normalised", NRM_TR.shape, f"{time.time()-t0:.0f}s")

az_bin = np.minimum((AZ_TR / (360 / AZ_STRAT_BINS)).astype(int), AZ_STRAT_BINS - 1)
strat = np.array([f"{a}_{b}" for a, b in zip(y, az_bin)])
if pd.Series(strat).value_counts().min() < N_FOLDS:
    strat = y
folds = np.zeros(len(train_df), int)
for f, (_, v) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(train_df, strat)):
    folds[v] = f
W = (len(y) / (2.0 * np.bincount(y)))[y]          # class-balance weights (metric is balanced accuracy)
print("folds", np.bincount(folds), "| class counts", np.bincount(y))

train (7854, 3) /kaggle/input/datasets/kawaljeetsinghbharaj/paradox/train_images/train_images | test (2000, 2) /kaggle/input/datasets/kawaljeetsinghbharaj/paradox/eval_images/eval_images
loaded (7854, 256, 256) (2000, 256, 256) 8s
azimuth-normalised (7854, 256, 256) 4s
folds [1571 1571 1571 1571 1570] | class counts [2854 5000]


## 3. Branch features

In [8]:
def hog_features(arr, ppc=16, size=224):
    def f(a):
        return hog(cv2.resize(a, (size, size), interpolation=cv2.INTER_AREA), orientations=9,
                   pixels_per_cell=(ppc, ppc), cells_per_block=(2, 2), block_norm="L2-Hys", feature_vector=True)
    with ThreadPoolExecutor(8) as ex:
        return np.stack(list(ex.map(f, arr))).astype(np.float32)

def load_lunar_fm():
    sys.path.insert(0, LFM_REPO) if LFM_REPO not in sys.path else None
    spec = importlib.util.spec_from_file_location("lunar_backbone", os.path.join(LFM_REPO, "terratorch_integration", "lunar_backbone.py"))
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    from huggingface_hub import hf_hub_download
    cfg = hf_hub_download(LFM_HF_REPO, "backbone/config.yaml", local_dir=LFM_WEIGHTS)
    ckpt = hf_hub_download(LFM_HF_REPO, "backbone/checkpoint.pt", local_dir=LFM_WEIGHTS)
    bb = mod.LunarBackbone(variant="base", modalities=["nac"], cfg=cfg, patch_size=16, checkpoint_path=ckpt)
    for n in ("decoder", "decoder_norm", "decoder_proj_context", "decoder_embeddings"):
        if hasattr(bb.model, n):
            delattr(bb.model, n)
    return bb.to(DEV).eval()

@torch.no_grad()
def embed(model, imgs, kind, bs=64, mean=None, std=None):
    out = []
    for i in range(0, len(imgs), bs):
        x = torch.from_numpy(imgs[i:i + bs]).to(DEV)[:, None].float().div_(255.0)
        with torch.autocast("cuda", dtype=torch.float16, enabled=(DEV == "cuda")):
            if kind == "lfm":
                outs = model((x - mean) / std)
                f = torch.cat([F.layer_norm(outs[k].mean(1), (outs[k].shape[-1],)) for k in (5, 8, 11)], 1)
            else:
                f = model(((x.expand(-1, 3, -1, -1) - mean) / std))
        out.append(f.float().cpu())
    return torch.cat(out).numpy()

FEATS = {}
t0 = time.time(); FEATS["hog_norm"] = (hog_features(NRM_TR), hog_features(NRM_TE)); print(f"hog_norm {FEATS['hog_norm'][0].shape} {time.time()-t0:.0f}s")
t0 = time.time(); FEATS["hog_raw"] = (hog_features(RAW_TR), hog_features(RAW_TE)); print(f"hog_raw  {FEATS['hog_raw'][0].shape} {time.time()-t0:.0f}s")

if USE_LFM:
    t0 = time.time()
    m = load_lunar_fm()
    mu = float(NRM_TR[::7].astype(np.float32).mean() / 255); sd = float(NRM_TR[::7].astype(np.float32).std() / 255)
    mean = torch.tensor(mu, device=DEV).view(1, 1, 1, 1); std = torch.tensor(sd, device=DEV).view(1, 1, 1, 1)
    FEATS["lfm"] = (embed(m, NRM_TR, "lfm", mean=mean, std=std), embed(m, NRM_TE, "lfm", mean=mean, std=std))
    del m; gc.collect(); torch.cuda.empty_cache()
    print(f"lfm      {FEATS['lfm'][0].shape} {time.time()-t0:.0f}s")

if USE_DINO:
    t0 = time.time()
    name = "convnext_small.dinov3_lvd1689m"
    m = timm.create_model(name, pretrained=True, num_classes=0).to(DEV).eval()
    dc = timm.data.resolve_model_data_config(m)
    mean = torch.tensor(dc["mean"], device=DEV).view(1, 3, 1, 1); std = torch.tensor(dc["std"], device=DEV).view(1, 3, 1, 1)
    FEATS["dino"] = (embed(m, NRM_TR, "timm", mean=mean, std=std), embed(m, NRM_TE, "timm", mean=mean, std=std))
    del m; gc.collect(); torch.cuda.empty_cache()
    print(f"dino     {FEATS['dino'][0].shape} {time.time()-t0:.0f}s")

hog_norm (7854, 6084) 59s
hog_raw  (7854, 6084) 58s


config.yaml:   0%|          | 0.00/12.1k [00:00<?, ?B/s]

backbone/checkpoint.pt: reconstructing file:   0%|          |  0.00B / 2.43GB            

backbone/checkpoint.pt: downloading bytes:           |  0.00B            

LunarBackbone: loaded modality info (20 modalities: ['aspect', 'aspect_3m', 'dtm', 'dtm_3m', 'metadata', 'nac', 'slope', 'slope_3m', 'static_maps', 'tok_aspect', 'tok_aspect_3m', 'tok_dtm', 'tok_dtm_3m', 'tok_nac', 'tok_slope', 'tok_slope_3m', 'tok_uv', 'tok_vis', 'uv', 'vis'])
encoder_embeddings.aspect.mod_emb
encoder_embeddings.aspect.pos_emb
encoder_embeddings.aspect.proj.weight
encoder_embeddings.aspect_3m.mod_emb
encoder_embeddings.aspect_3m.pos_emb
encoder_embeddings.aspect_3m.proj.weight
encoder_embeddings.dtm.mod_emb
encoder_embeddings.dtm.pos_emb
encoder_embeddings.dtm.proj.weight
encoder_embeddings.dtm_3m.mod_emb
encoder_embeddings.dtm_3m.pos_emb
encoder_embeddings.dtm_3m.proj.weight
encoder_embeddings.metadata.mod_emb
encoder_embeddings.metadata.pos_emb
encoder_embeddings.metadata.token_emb.weight
encoder_embeddings.slope.mod_emb
encoder_embeddings.slope.pos_emb
encoder_embeddings.slope.proj.weight
encoder_embeddings.slope_3m.mod_emb
encoder_embeddings.slope_3m.pos_emb
encod

model.safetensors: reconstructing file:   0%|          |  0.00B /  198MB            

model.safetensors: downloading bytes:           |  0.00B            

dino     (7854, 768) 61s


## 4. Per-branch cross-validation (same folds everywhere)

In [9]:
def best_threshold(yy, p):
    grid = np.linspace(0.02, 0.98, 193)
    s = [balanced_accuracy_score(yy, (p >= t).astype(int)) for t in grid]
    i = int(np.argmax(s)); return float(grid[i]), float(s[i])

def cv_branch(Xtr, Xte, C=0.05, n_pca=256):
    oof, test_p = np.zeros(len(y)), np.zeros(len(Xte))
    for f in range(N_FOLDS):
        a, b = folds != f, folds == f
        sc = StandardScaler().fit(Xtr[a])
        Xa, Xb, Xt = sc.transform(Xtr[a]), sc.transform(Xtr[b]), sc.transform(Xte)
        if n_pca and Xtr.shape[1] > n_pca:
            pca = PCA(n_pca, random_state=SEED).fit(Xa)
            Xa, Xb, Xt = pca.transform(Xa), pca.transform(Xb), pca.transform(Xt)
        clf = LogisticRegression(C=C, max_iter=4000).fit(Xa, y[a], sample_weight=W[a])
        oof[b] = clf.predict_proba(Xb)[:, 1]
        test_p += clf.predict_proba(Xt)[:, 1] / N_FOLDS
    return oof, test_p

OOF, TEST, rows = {}, {}, []
for name, (Xtr, Xte) in FEATS.items():
    t0 = time.time()
    OOF[name], TEST[name] = cv_branch(Xtr, Xte)
    thr, ba = best_threshold(y, OOF[name])
    rows.append({"branch": name, "auc": roc_auc_score(y, OOF[name]),
                 "ba@0.5": balanced_accuracy_score(y, (OOF[name] >= 0.5).astype(int)), "ba@thr": ba, "thr": thr})
    print(f"{name:9s} AUC {rows[-1]['auc']:.4f} BA@thr {ba:.4f} ({time.time()-t0:.0f}s)")
print(pd.DataFrame(rows).to_string(index=False))

hog_norm  AUC 0.7375 BA@thr 0.6959 (34s)
hog_raw   AUC 0.4601 BA@thr 0.5006 (33s)
lfm       AUC 0.7354 BA@thr 0.6932 (21s)
dino      AUC 0.7140 BA@thr 0.6800 (13s)
  branch      auc   ba@0.5   ba@thr   thr
hog_norm 0.737491 0.692594 0.695912 0.575
 hog_raw 0.460138 0.471045 0.500602 0.230
     lfm 0.735356 0.692322 0.693226 0.550
    dino 0.713977 0.679339 0.680018 0.505


In [10]:
# ---- multi-scale HOG, centre-crop HOG, sun-relative gradient features ----
t0 = time.time()
FEATS["hog_fine"] = (hog_features(NRM_TR, ppc=8, size=128), hog_features(NRM_TE, ppc=8, size=128))
FEATS["hog_coarse"] = (hog_features(NRM_TR, ppc=32, size=224), hog_features(NRM_TE, ppc=32, size=224))
centre = lambda a: a[:, 64:192, 64:192]
FEATS["hog_centre"] = (hog_features(centre(NRM_TR)), hog_features(centre(NRM_TE)))
print(f"extra HOG branches in {time.time()-t0:.0f}s")

def sun_relative(arr, n_rings=4, n_bins=12):
    # after normalisation the sun is a fixed direction, so gradient orientation carries the shadow order
    H = arr.shape[1]
    yy, xx = np.mgrid[0:H, 0:H] - H / 2
    rad = np.sqrt(yy ** 2 + xx ** 2) / (H / 2)
    ring = np.clip((rad * n_rings).astype(int), 0, n_rings - 1)
    def f(a):
        a = a.astype(np.float32)
        gx = cv2.Sobel(a, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(a, cv2.CV_32F, 0, 1, ksize=3)
        mag = np.hypot(gx, gy)
        ang = (np.arctan2(gy, gx) + 2 * np.pi) % (2 * np.pi)      # 0 = +x, measured in the normalised frame
        b = np.clip((ang / (2 * np.pi) * n_bins).astype(int), 0, n_bins - 1)
        out = []
        for r in range(n_rings):
            m = ring == r
            h = np.bincount(b[m], weights=mag[m], minlength=n_bins)
            out.append(h / (h.sum() + 1e-6))
            out.append([a[m].mean() / 255, a[m].std() / 255, gy[m].mean(), gx[m].mean()])
        return np.concatenate([np.asarray(o).ravel() for o in out]).astype(np.float32)
    with ThreadPoolExecutor(8) as ex:
        return np.stack(list(ex.map(f, arr)))

t0 = time.time()
FEATS["sunrel"] = (sun_relative(NRM_TR), sun_relative(NRM_TE))
print(f"sunrel {FEATS['sunrel'][0].shape} in {time.time()-t0:.0f}s")

extra HOG branches in 147s
sunrel (7854, 64) in 45s


In [11]:
# ---- Lunar FM at patch size 8 (FlexiViT, 1024 tokens) + flip-TTA, and DINOv3 ViT-B ----
def load_lunar_fm_ps(ps):
    spec = importlib.util.spec_from_file_location(
        "lunar_backbone", os.path.join(LFM_REPO, "terratorch_integration", "lunar_backbone.py"))
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    from huggingface_hub import hf_hub_download
    cfg = hf_hub_download(LFM_HF_REPO, "backbone/config.yaml", local_dir=LFM_WEIGHTS)
    ckpt = hf_hub_download(LFM_HF_REPO, "backbone/checkpoint.pt", local_dir=LFM_WEIGHTS)
    bb = mod.LunarBackbone(variant="base", modalities=["nac"], cfg=cfg, patch_size=ps, checkpoint_path=ckpt)
    for n in ("decoder", "decoder_norm", "decoder_proj_context", "decoder_embeddings"):
        if hasattr(bb.model, n):
            delattr(bb.model, n)
    return bb.to(DEV).eval()

def embed_tta(model, imgs, kind, mean, std, bs=32):
    a = embed(model, imgs, kind, bs=bs, mean=mean, std=std)
    b = embed(model, imgs[:, :, ::-1].copy(), kind, bs=bs, mean=mean, std=std)   # mirror across the sun axis
    return 0.5 * (a + b)

mu = float(NRM_TR[::7].astype(np.float32).mean() / 255); sd = float(NRM_TR[::7].astype(np.float32).std() / 255)
mean_l = torch.tensor(mu, device=DEV).view(1, 1, 1, 1); std_l = torch.tensor(sd, device=DEV).view(1, 1, 1, 1)

try:
    t0 = time.time(); m = load_lunar_fm_ps(8)
    FEATS["lfm_ps8"] = (embed_tta(m, NRM_TR, "lfm", mean_l, std_l), embed_tta(m, NRM_TE, "lfm", mean_l, std_l))
    del m; gc.collect(); torch.cuda.empty_cache()
    print(f"lfm_ps8 {FEATS['lfm_ps8'][0].shape} in {time.time()-t0:.0f}s")
except Exception as e:
    print("lfm_ps8 skipped:", type(e).__name__, e)

try:
    t0 = time.time()
    m = timm.create_model("vit_base_patch16_dinov3.lvd1689m", pretrained=True, num_classes=0, img_size=IMG_SIZE).to(DEV).eval()
    dc = timm.data.resolve_model_data_config(m)
    mn = torch.tensor(dc["mean"], device=DEV).view(1, 3, 1, 1); sdv = torch.tensor(dc["std"], device=DEV).view(1, 3, 1, 1)
    FEATS["dino_vit"] = (embed_tta(m, NRM_TR, "timm", mn, sdv), embed_tta(m, NRM_TE, "timm", mn, sdv))
    del m; gc.collect(); torch.cuda.empty_cache()
    print(f"dino_vit {FEATS['dino_vit'][0].shape} in {time.time()-t0:.0f}s")
except Exception as e:
    print("dino_vit skipped:", type(e).__name__, e)

LunarBackbone: loaded modality info (20 modalities: ['aspect', 'aspect_3m', 'dtm', 'dtm_3m', 'metadata', 'nac', 'slope', 'slope_3m', 'static_maps', 'tok_aspect', 'tok_aspect_3m', 'tok_dtm', 'tok_dtm_3m', 'tok_nac', 'tok_slope', 'tok_slope_3m', 'tok_uv', 'tok_vis', 'uv', 'vis'])
encoder_embeddings.aspect.mod_emb
encoder_embeddings.aspect.pos_emb
encoder_embeddings.aspect.proj.weight
encoder_embeddings.aspect_3m.mod_emb
encoder_embeddings.aspect_3m.pos_emb
encoder_embeddings.aspect_3m.proj.weight
encoder_embeddings.dtm.mod_emb
encoder_embeddings.dtm.pos_emb
encoder_embeddings.dtm.proj.weight
encoder_embeddings.dtm_3m.mod_emb
encoder_embeddings.dtm_3m.pos_emb
encoder_embeddings.dtm_3m.proj.weight
encoder_embeddings.metadata.mod_emb
encoder_embeddings.metadata.pos_emb
encoder_embeddings.metadata.token_emb.weight
encoder_embeddings.slope.mod_emb
encoder_embeddings.slope.pos_emb
encoder_embeddings.slope.proj.weight
encoder_embeddings.slope_3m.mod_emb
encoder_embeddings.slope_3m.pos_emb
encod

model.safetensors: reconstructing file:   0%|          |  0.00B /  343MB            

model.safetensors: downloading bytes:           |  0.00B            

dino_vit (7854, 768) in 112s


## 5. Fold-safe stack + azimuth interactions

In [12]:
# ---- cross-validate any branch not scored yet, with a small C sweep ----
def fourier(az, K=2):
    r = np.deg2rad(az)
    return np.concatenate([np.stack([np.sin(k * r), np.cos(k * r)], 1) for k in range(1, K + 1)], 1)

BEST_C = globals().get("BEST_C", {})
for name, (Xtr, Xte) in FEATS.items():
    if name in OOF:
        continue
    t0, best_c, best_ba = time.time(), None, -1
    for C in (0.01, 0.05, 0.2, 1.0):
        o, tp = cv_branch(Xtr, Xte, C=C)
        ba = best_threshold(y, o)[1]
        if ba > best_ba:
            best_ba, best_c, OOF[name], TEST[name] = ba, C, o, tp
    BEST_C[name] = best_c
    print(f"{name:10s} AUC {roc_auc_score(y, OOF[name]):.4f} BA@thr {best_ba:.4f} (C={best_c}, {time.time()-t0:.0f}s)")

# ---- stacking: logistic regression and LightGBM meta-models, image branches only ----
import lightgbm as lgb
names = list(OOF)
P_tr = np.stack([OOF[n] for n in names], 1); P_te = np.stack([TEST[n] for n in names], 1)
A_tr, A_te = fourier(AZ_TR), fourier(AZ_TE)

def cv_lgbm(Xtr, Xte, **kw):
    oof, tp = np.zeros(len(y)), np.zeros(len(Xte))
    for f in range(N_FOLDS):
        a, b = folds != f, folds == f
        m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, subsample=0.8,
                               colsample_bytree=0.8, class_weight="balanced", verbose=-1, random_state=SEED, **kw)
        m.fit(Xtr[a], y[a])
        oof[b] = m.predict_proba(Xtr[b])[:, 1]; tp += m.predict_proba(Xte)[:, 1] / N_FOLDS
    return oof, tp

cands = {
    "stack LR: image only": cv_branch(P_tr, P_te, C=1.0, n_pca=0),
    "stack LGBM: image only": cv_lgbm(P_tr, P_te),
    "mean of branches": (P_tr.mean(1), P_te.mean(1)),
    "reference: azimuth only": cv_branch(A_tr, A_te, C=1.0, n_pca=0),
}
S_tr = np.concatenate([P_tr, A_tr, P_tr * A_tr[:, :1], P_tr * A_tr[:, 1:2]], 1)
S_te = np.concatenate([P_te, A_te, P_te * A_te[:, :1], P_te * A_te[:, 1:2]], 1)
cands["stack: image + azimuth"] = cv_branch(S_tr, S_te, C=1.0, n_pca=0)
np.save(f"{OUT_DIR}/stack_oof.npy", np.stack([OOF[n] for n in names]))

for n in names:
    cands[f"branch: {n}"] = (OOF[n], TEST[n])
res = pd.DataFrame([{"model": k, "auc": roc_auc_score(y, o), "ba@thr": best_threshold(y, o)[1],
                     "thr": best_threshold(y, o)[0]} for k, (o, t) in cands.items()]).sort_values("ba@thr", ascending=False)
print(res.to_string(index=False))

img_only = [r for r in res.model if r.startswith(("stack", "mean", "branch"))][0]
best = res[res.model == img_only].iloc[0]
o, t = cands[img_only]
pd.DataFrame({"image_id": test_df.image_id, "label": (t >= best["thr"]).astype(int)}).to_csv(f"{OUT_DIR}/submission_stack.csv", index=False)
print(f"\nbest image model: {img_only} | OOF BA {best['ba@thr']:.4f} -> submission_stack.csv")

hog_fine   AUC 0.7383 BA@thr 0.6965 (C=0.01, 158s)
hog_coarse AUC 0.7389 BA@thr 0.7022 (C=0.01, 53s)
hog_centre AUC 0.6826 BA@thr 0.6447 (C=0.05, 133s)
sunrel     AUC 0.7240 BA@thr 0.6789 (C=0.05, 2s)
lfm_ps8    AUC 0.6547 BA@thr 0.6294 (C=0.01, 110s)
dino_vit   AUC 0.6645 BA@thr 0.6442 (C=0.01, 51s)
                  model      auc   ba@thr   thr
reference: azimuth only 0.786767 0.782104 0.445
 stack: image + azimuth 0.788885 0.773623 0.470
   stack LR: image only 0.757390 0.721782 0.520
 stack LGBM: image only 0.748745 0.711715 0.495
       mean of branches 0.748390 0.710012 0.525
     branch: hog_coarse 0.738937 0.702227 0.545
       branch: hog_fine 0.738273 0.696522 0.545
       branch: hog_norm 0.737491 0.695912 0.575
            branch: lfm 0.735356 0.693226 0.550
           branch: dino 0.713977 0.680018 0.505
         branch: sunrel 0.724041 0.678915 0.525
     branch: hog_centre 0.682628 0.644685 0.525
       branch: dino_vit 0.664488 0.644208 0.550
        branch: lfm_ps8 0.

In [13]:
import joblib
models = {}
for name, (Xtr, Xte) in FEATS.items():
    C = BEST_C.get(name, 0.05)
    sc = StandardScaler().fit(Xtr); Xs = sc.transform(Xtr)
    pca = PCA(256, random_state=SEED).fit(Xs) if Xtr.shape[1] > 256 else None
    if pca is not None:
        Xs = pca.transform(Xs)
    models[name] = {"scaler": sc, "pca": pca, "C": C,
                    "clf": LogisticRegression(C=C, max_iter=4000).fit(Xs, y, sample_weight=W)}

sc_p = StandardScaler().fit(P_tr)
models["stack_image_only"] = {"scaler": sc_p, "branch_order": names,
                              "clf": LogisticRegression(C=1.0, max_iter=4000).fit(sc_p.transform(P_tr), y, sample_weight=W)}
models["_meta"] = {"branches": list(FEATS), "folds_seed": SEED, "n_folds": N_FOLDS,
                   "submitted_model": img_only, "threshold": float(best["thr"]), "oof_ba": float(best["ba@thr"]),
                   "img_size": IMG_SIZE, "rot_crop": ROT_CROP, "rotation": "-sun_azimuth_angle"}
joblib.dump(models, f"{OUT_DIR}/stack_weights.joblib")
print("saved stack_weights.joblib:", list(models))

saved stack_weights.joblib: ['hog_norm', 'hog_raw', 'lfm', 'dino', 'hog_fine', 'hog_coarse', 'hog_centre', 'sunrel', 'lfm_ps8', 'dino_vit', 'stack_image_only', '_meta']
